In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer= 6,
    heads= 4,
    embed_dim= 256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=50,
    std_coeff=25,
    cov_coeff=1,
    pert_latent_dim=128,
    pert_mode_dim=64,
)


EVAL_BATCH_SIZE = 32

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 7,979,650
ACpredictor: 9,987,072
PerturbationComposer: 444,224


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [6]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
pt_eval_results = run_encoder_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

Using cuda
found 135 shards for split test


batch_invariance: Extracting embeddings: 100%|█████████| 10800/10800 [06:37<00:00, 27.14it/s]


Training classifiers...
batch_invariance: Batch=0.0521 (22.5x), Pert=0.0281 (31.1x)
batch_invariance summary: global_ratio=0.540, within_dataset_macro_ratio=0.462
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
gene_embedding_pathways: KEGG ratio=1.1854
essential_gene_prediction: Pearson=0.1228, AUROC=0.5797
found 135 shards for split test


cell_type_probing: Extracting embeddings: 100%|████████| 10800/10800 [06:33<00:00, 27.46it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.8519 (3.4x chance), Macro F1=0.5660
found 135 shards for split test


reconstruction: Extracting embeddings: 100%|███████████████████| 3/3 [00:00<00:00, 10.81it/s]


Training reconstruction MLP...
reconstruction: MSE=0.0322, Pearson R=0.9541
found 135 shards for split test


perturbation_detection: Extracting embeddings: 100%|███| 10800/10800 [12:43<00:00, 14.14it/s]


Training perturbation detector...
perturbation_detection: AUROC=0.5610, Accuracy=0.5404
found 135 shards for split test


embedding_consistency: Extracting embeddings:  61%|███  | 6594/10800 [04:01<02:27, 28.43it/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [7]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [8]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

Using cuda
Loaded 10791 v0.7 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 128])
Encoded chemical sequences: torch.Size([188, 128])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 128])
seq_to_target_retrieval: dna_mrr=0.0257
cross_modality_target_consistency: Within=0.5934, Between=0.5033, Ratio=1.18x
seq_target_gap_analysis: dna_gap=0.87
paired_alignment_quality: dna_sim=0.6051
mode_sensitivity: Classification_acc=0.8048 (5.6x chance)
mode_semantic_consistency: semantic_gap=-0.0386, cross_mode_mrr=0.0833
fusion_quality: Fused_var=0.5003, Seq_var=0.3885, Target_var=0.0660
missing_data_robustness: Fused_MRR=0.6176, Seq_only=0.0288, Target_only=0.9975
found 135 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|██████| 500/500 [00:02<00:00, 242.12it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0851, target_only=0.1966, fused=0.2226
action_vector_pathways DNA: ratio=1.0013760208784863
Saved report to /home/ubuntu/data/v0_7/eval_results/composer_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.02565720296922547,
    'median_rank': 938.0,
    'mean_rank': 1775.4471307619945,
    'n_queries': 10630,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.00940733772342427,
     '5': 0.03452492944496707,
     '10': 0.05155221072436501,
     '20': 0.07883349012229539,
     '50': 0.13198494825964252}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 921,
   'n_within_pairs': 1408,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.5934078097343445,
   'between_target_sim': 0.5032951541304588,
   'consistency_ratio': 1.1790453471771907}},
 'seq_target_gap_analysis': {'target_variance': 8.14721393585205,
  'n_targets': 9975,
  'dna': {'seq_variance': 51.759422302246094,
   'centroid_distance': 4.470981597900391,
   'mean_within_seq': 10.138100968110354,
   'mean_seq_to_target': 8.836633682250977,
   'gap_ratio': 0.8716261270278156,
   'n_sequences': 11643}}

In [9]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [10]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [11]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Using cuda
found 135 shards for split test


Running test inference:   0%|                                      | 0/10800 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:   1%|▏                          | 81/10800 [00:24<6:09:13,  2.07s/it]/home/ubuntu/code/biojepa/evals/evals.py:643: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference:  28%|███████▍                   | 2997/10800 [12:34<18:40,  6.96it/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()